# NB-06 — LM-Weight Sweep (the mechanism experiment)

RQ1 established **that** models with less data exposure are easier to attack. This notebook tests
**why**.

The hypothesis: what protects a high-exposure model is its **linguistic prior** — its sense of
which Bengali characters and words follow which. A model that has seen 32,000 distinct clips has
a stronger one than a model that has seen 809 clips ninety-nine times.

If that is right, then supplying an *external* language model should make the attack harder, and
should help the low-exposure model **more**, because it has the least of its own.

So we take one acoustic model, hold it completely fixed, and turn a language model up and down
like a volume knob.

---

### Why this is cheap

The LM is applied at **decode** time, not inside the attack loop. So the expensive part — PGD —
runs **once per model**. The α sweep is then just re-decoding audio we already have.

Roughly 45 minutes, not the 3 hours a naive design would cost.

### The honest caveat, stated up front

The attacker here optimises against **greedy decoding** and does not know an LM will be applied.
That is a *non-adaptive* threat model, and a reviewer will say so. Two responses, both in the
notebook:

1. As a **mechanism probe** it is still valid — α varies linguistic knowledge while the acoustic
   model is byte-identical, so any change in attack success is attributable to the prior alone.
2. As a **defence** claim it is an upper bound. An adaptive attacker would do better. Section 9
   quantifies how much headroom the attacker has.

### Kaggle settings

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 ×2** |
| Internet | On |
| Persistence | Files only |
| **Inputs** | `bn-asr-splits` · `bengaliai-speech` · `ckpt-xlsr-1h` · `ckpt-xlsr-100h` · `clean-wer-results` |
| Optional inputs | `lm-sweep-results` (to re-sweep from cached logits, §11) |
| Output | `lm-sweep-results` |

## 1 · Configuration

In [ ]:
# Models to sweep. The first two are the ISO-WER PAIR from NB-02: matched clean
# WER, ~40x difference in distinct audio seen. If the LM helps the 1h model more
# than its accuracy-matched twin, that is the mechanism claim in one comparison.
MODELS = [
    ("1h-final",      1,   "final"),          # attack-set WER 0.6471
    ("100h-ms1000",   100, "milestone-1000"), # attack-set WER 0.6667  <- matched
    ("100h-final",    100, "final"),          # attack-set WER 0.2500
]

# Extended to 8.0 on 2026-09-26. At the original ceiling of 3.0 the 1h-final
# optimum sat AT the edge, so its curve had not turned and alpha* was unknown.
ALPHAS   = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]   # LM weight
BETA     = 0.0          # length bonus. The LM score is now mean-centred (section 4),
                        # so it carries no per-character tax for BETA to cancel, and
                        # alpha=0 is EXACTLY LM-free beam search. See BETA_GRID below.
BETA_GRID = [0.0, 0.25, 0.5, 1.0]   # robustness check, re-run from cached logits, no GPU
ATTACK_EPS = 1.0e-3     # fixed budget; sits between the rung medians from NB-05
ATTACK_STEPS = 800      # same as NB-05's CONFIRM_STEPS
N_EVAL   = 25
BATCH_SIZE = 8
USE_AMP  = True

LM_ORDER   = 5          # character n-gram order
LM_BACKOFF = 0.4        # stupid-backoff discount
LM_TEXTS   = 200_000    # transcripts used to train the LM
LM_HELD    = 5_000      # disjoint transcripts used to measure the LM's entropy
LM_MIN_COUNT = 2        # prune singleton contexts (memory)

BEAM_WIDTH = 48         # was 16 in the 2026-09-23 run: too narrow to realise most
BEAM_TOPK  = 10         # was 6.  of the LM's benefit. Costs ~8 min of CPU, once.

SAVE_LOGITS = True      # persist CTC logits so the whole sweep re-runs with no GPU

# CPU re-sweep. True = skip PGD entirely, load cached logits from lm-sweep-results,
# and re-run both gates and the alpha sweep on CPU. Use it to widen ALPHAS, try
# BETA_GRID, or change the decoder without spending another 70 GPU-minutes.
# Set Accelerator to None and attach lm-sweep-results.
CPU_RESWEEP = None      # None = auto: CPU session -> re-sweep, GPU session -> full run

SESSION_HOURS = 8.0


## 2 · Boilerplate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys, json, time, re, glob, math, random, unicodedata
from collections import defaultdict, Counter
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP = "/kaggle/working", "/kaggle/temp"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(f"{TEMP}/scratch", exist_ok=True)
if CPU_RESWEEP is None:          # no GPU means you came here to re-sweep
    CPU_RESWEEP = not torch.cuda.is_available()
    print(f"CPU_RESWEEP auto-set to {CPU_RESWEEP} "
          f"({'no GPU -- loading cached logits' if CPU_RESWEEP else 'GPU present -- full run'})")
assert CPU_RESWEEP or torch.cuda.is_available(), (
    "Settings -> Accelerator -> GPU T4 x2, or set CPU_RESWEEP=True")
T0 = time.time()

PROV = {"notebook": "NB-06-lm-weight-sweep", "seed": SEED, "alphas": ALPHAS,
        "beta": BETA, "attack_eps": ATTACK_EPS, "attack_steps": ATTACK_STEPS,
        "lm_order": LM_ORDER, "lm_texts": LM_TEXTS, "beam_width": BEAM_WIDTH,
        "use_amp": USE_AMP, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "torch": torch.__version__, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
import numpy as _np
NP_BEFORE = _np.__version__
!pip install -q transformers==4.44.2 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2
import subprocess
NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}"
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, jiwer
print(f"transformers {transformers.__version__} | numpy {NP_BEFORE}")
PROV["transformers"] = transformers.__version__; PROV["numpy"] = NP_BEFORE

## 3 · Inputs

In [ ]:
def list_inputs():
    out = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        out += [p for p in glob.glob(pat) if os.path.isdir(p)]
    return sorted(set(out))
INPUTS = list_inputs()

def find_input(pattern, what):
    hits = [p for p in INPUTS if re.search(pattern, os.path.basename(p), re.I)]
    if not hits:
        print("Visible under /kaggle/input:"); [print("   ", p) for p in INPUTS]
        raise AssertionError(f"Could not find {what} (regex {pattern})")
    return max(hits, key=lambda p: (p.count("/"), len(p)))

def model_dir(rung, ckpt):
    root = find_input(rf"ckpt-xlsr-{rung}h$", f"ckpt-xlsr-{rung}h")
    d = (root if os.path.exists(f"{root}/config.json") else f"{root}/final") \
        if ckpt == "final" else f"{root}/milestones/step-{ckpt.split('-')[1]}"
    assert os.path.exists(f"{d}/config.json"), f"no config.json under {d}"
    return d

SPLITS = find_input(r"splits", "bn-asr-splits")
ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

ev = pd.read_csv(f"{SPLITS}/attack_eval_{N_EVAL}.csv")
probe = str(ev.iloc[0]["path"])
if not os.path.exists(probe):
    stem = os.path.basename(os.path.dirname(probe)); ext = os.path.splitext(probe)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    ev["path"] = cands[0] + "/" + ev["utt_id"].astype(str) + ext
assert os.path.exists(str(ev.iloc[0]["path"])), "attack audio unresolved"
ev["text"] = ev["text"].map(norm_bn)
print(f"attack set: {len(ev)} utterances")

for name, rung, ckpt in MODELS:
    print(f"  {name:<14} {model_dir(rung, ckpt)}")

### 3b · Verify the iso-WER pair against NB-02

Section 9's entire claim is a comparison between two models *of matched clean accuracy*.
That matching was established by NB-02, but `MODELS` records it only in a code comment.
This cell turns it into an assertion, keyed on model directory rather than dataset name,
and fails in seconds rather than after an hour of GPU if the pair has drifted.

It also gives section 8 an independent reference: NB-02's **greedy** attack-set WER for the
same 25 utterances, against which this notebook's `α=0` beam baseline can be cross-checked.


In [ ]:
# ---------------------------------------------------------------------------
# VERIFY THE ISO-WER PAIR AGAINST NB-02, before spending an hour of GPU.
#
# MODELS hardcodes the accuracy-matched pair and carries the matching WERs in
# code comments. A comment is not a check. If NB-02 is ever re-run and its
# numbers move, section 9 would compare the wrong two models and the headline
# would be wrong in a way nothing else in this notebook could detect.
#
# Rows are matched on MODEL DIRECTORY, never on dataset name -- see the
# 2026-09-23 naming note in RUNLOG.
# ---------------------------------------------------------------------------
ISO_PAIR = ("1h-final", "100h-ms1000")   # the accuracy-matched pair
ISO_TOL  = 0.05                          # NB-02 reports a gap of 0.0196

CLEAN = next((p for p in INPUTS
              if os.path.exists(f"{p}/clean_wer_all_checkpoints.csv")), None)
NB02_WER = {}
if CLEAN is None:
    print("*** clean-wer-results NOT attached ***")
    print("    The iso-WER pair is taken on trust from a code comment and cannot")
    print("    be verified, and section 8 cannot cross-check its alpha=0 baseline")
    print("    against NB-02. Attach it before the run that feeds the thesis.")
else:
    nb02 = pd.read_csv(f"{CLEAN}/clean_wer_all_checkpoints.csv")
    nb02["rp"] = nb02["path"].map(os.path.realpath)
    for name, rung, ckpt in MODELS:
        hit = nb02[nb02.rp == os.path.realpath(model_dir(rung, ckpt))]
        if len(hit) == 0:          # mount point moved between sessions
            if ckpt == "final":
                hit = nb02[(nb02.rung == rung) & (nb02.kind == "final")]
            else:
                hit = nb02[(nb02.rung == rung) & (nb02.step == int(ckpt.split("-")[1]))]
        assert len(hit) == 1, f"{name}: {len(hit)} matching NB-02 rows, expected 1"
        NB02_WER[name] = float(hit.iloc[0]["wer_attack"])
        print(f"  NB-02 attack-set WER   {name:<14} {NB02_WER[name]:.4f}")

    lo, hi = ISO_PAIR
    gap = abs(NB02_WER[lo] - NB02_WER[hi])
    assert gap < ISO_TOL, (
        f"ISO-WER PAIR BROKEN: {lo} {NB02_WER[lo]:.4f} vs {hi} {NB02_WER[hi]:.4f} "
        f"(gap {gap:.4f} >= {ISO_TOL}). Re-select the pair from NB-02's "
        f"iso_wer_pairs.csv before running section 9.")
    print(f"\n  iso-WER pair verified: {lo} vs {hi}, gap {gap:.4f} < {ISO_TOL}")
PROV["nb02_attack_wer"] = NB02_WER
PROV["iso_pair"] = list(ISO_PAIR)
PROV["nb02_verified"] = CLEAN is not None


## 4 · Build a Bengali character n-gram language model

Trained on the **training-split transcripts only** — never on dev or test, and never on the
attacked utterances. Text is the cheap resource here, which is the practical point: an external
LM costs nothing next to collecting more audio.

Stupid backoff rather than Kneser-Ney, deliberately: it is a dozen lines, has no external
dependency to compile on Kaggle, and the experiment only needs the LM to be *monotonically
better* as α rises, not state of the art.

### Two things this cell does that the 2026-09-23 run did not

**It records the LM's alphabet**, so `check_lm_alphabet()` can assert that every symbol the
decoder is able to emit is one the LM has actually seen. A symbol the LM never saw scores at the
backoff floor (`log(1e-9)` plus four backoff penalties, about **−25 nats**) every single time it
appears, so the decoder simply stops emitting it — silently, and the result looks like a finding.
That is exactly what happened on 2026-09-23: the LM was trained on `norm_bn()` output, where
words are separated by a **space**, while the decoder scored the wav2vec2 CTC word-delimiter
token `|`. Every word boundary cost −25α, all boundaries were suppressed, and clean WER pinned at
**exactly 1.0** while CER stayed low — because the characters were right and only the spaces were
gone.

**It mean-centres the score** by measuring the LM's entropy on held-out text and adding it back
as an offset. Raw `log P` averages about −1.5 nats per character, so an uncentred LM charges
every emitted character a flat −1.5α tax; `BETA` was there to cancel it but could not, because
`BETA` is fixed while α scales. Centring makes α scale only how far a character deviates from an
average Bengali character, which is the quantity the experiment is actually about.


In [ ]:
class CharNgramLM:
    """Character n-gram with stupid backoff. Self-contained: no KenLM build needed.

    `offset` is added to every score. Set it to the model's mean -logP/char on held-out
    text and the score becomes zero-mean, so alpha scales how far a character deviates
    from an average one rather than imposing a flat per-character tax."""
    def __init__(self, order=LM_ORDER, backoff=LM_BACKOFF):
        self.order = order
        self.log_backoff = math.log(backoff)
        self.counts = defaultdict(Counter)   # context -> Counter(next_char)
        self.totals = {}
        self.floor = math.log(1e-9)   # replaced at train() with log(1/|alphabet|)
        self.offset = 0.0
        self.alphabet = set()

    def train(self, texts, min_count=LM_MIN_COUNT):
        pad = "\x02" * (self.order - 1)
        for t in texts:
            s = pad + t + "\x03"
            for i in range(self.order - 1, len(s)):
                ch = s[i]
                for o in range(0, self.order):
                    self.counts[s[i-o:i]][ch] += 1
        if min_count > 1:                      # prune rare contexts to bound memory
            for ctx in [c for c, d in self.counts.items()
                        if c and sum(d.values()) < min_count]:
                del self.counts[ctx]
        self.totals = {c: sum(d.values()) for c, d in self.counts.items()}
        self.alphabet = set(self.counts[""])   # every symbol the LM has ever seen
        # A floor of log(1e-9) = -20.7 against a ~60-symbol alphabet makes one
        # unseen symbol cost more than a whole word of real text, so a single
        # mismatched symbol can flip the decoder's entire search. Uniform is the
        # honest floor: "never seen" should mean "no information", not "impossible".
        self.floor = math.log(1.0 / (len(self.alphabet) + 1))
        return self

    def logp(self, ctx, ch):
        """log P(ch | ctx) with stupid backoff, shifted by self.offset."""
        ctx = ctx[-(self.order - 1):]
        pen = 0.0
        for o in range(len(ctx), -1, -1):
            c = ctx[len(ctx)-o:] if o else ""
            d = self.counts.get(c)
            if d is not None:
                n = d.get(ch, 0)
                if n > 0:
                    return math.log(n / self.totals[c]) + pen + self.offset
            pen += self.log_backoff
        return self.floor + pen + self.offset


BOS = "\x02" * (LM_ORDER - 1)   # the left context the LM was trained with
EOS = "\x03"


def check_lm_alphabet(id2char, lm, texts, name=""):
    """GATE. Can a CORRECT transcription be both emitted and scored?

    The 2026-09-23 failure was not "some symbol is unknown to the LM" -- it was
    that the symbol carrying every word boundary was unknown, so boundaries were
    suppressed and WER pinned at exactly 1.0. The test that catches that, without
    tripping over genuinely rare characters, is anchored on the reference text:

      1. every character the references need must be EMITTABLE by the decoder.
         With the delimiter unmapped, id2char holds '|' and no ' ', while the
         references are full of spaces -- this fires immediately.
      2. every character the references need must be SCORABLE by the LM.
      3. anything else the decoder can emit but the LM has not seen is reachable
         only on a WRONG path. The floor is the honest score for it and
         suppressing it is correct behaviour, so report and continue.
    """
    dec, need = set(id2char.values()), set("".join(texts))
    miss_dec = sorted(need - dec)
    assert not miss_dec, (
        f"{name}: the references require {miss_dec!r}, which the decoder cannot emit. "
        f"If ' ' is in that list the CTC word delimiter is unmapped: every word "
        f"boundary would score at the floor and be suppressed (see 2026-09-23).")
    miss_lm = sorted(need - lm.alphabet)
    assert not miss_lm, (
        f"{name}: the references contain {miss_lm!r}, absent from the LM's "
        f"{len(lm.alphabet)}-symbol alphabet. The LM cannot score a correct "
        f"transcription, so raise LM_TEXTS or widen the LM's training text.")
    extra = sorted(dec - lm.alphabet)
    if extra:
        print(f"  {name}: {len(extra)} rare symbol(s) the LM never saw {extra!r} -- "
              f"absent from all references, so reachable only on wrong paths. "
              f"Scored at the floor ({lm.floor:.1f} nats). Continuing.")
    else:
        print(f"  {name}: decoder alphabet fully covered by the LM")
    return extra


man = pd.read_csv(f"{SPLITS}/manifest_full.csv", usecols=["text", "split"])
train_text = man[man.split == "train"]["text"].dropna().map(norm_bn)
train_text = train_text[train_text.str.len() > 0]
print(f"training transcripts available: {len(train_text)}")
assert len(train_text) >= LM_TEXTS + LM_HELD, "not enough train text for a held-out slice"
shuf_idx = train_text.sample(frac=1.0, random_state=SEED)
sample = shuf_idx.iloc[:LM_TEXTS].tolist()                     # LM training text
held   = shuf_idx.iloc[LM_TEXTS:LM_TEXTS + LM_HELD].tolist()   # disjoint, for entropy
del man, train_text, shuf_idx

t0 = time.time()
LM = CharNgramLM().train(sample)
print(f"LM trained on {len(sample)} transcripts in {time.time()-t0:.0f}s")
print(f"  contexts: {len(LM.counts)} | alphabet: {len(LM.alphabet)} symbols")
PROV["lm_contexts"] = len(LM.counts)

def score_text(s):                      # mean logP per character, with BOS context
    s2 = BOS + s
    return sum(LM.logp(s2[max(0, i-LM_ORDER+1):i], s2[i])
               for i in range(LM_ORDER-1, len(s2))) / max(1, len(s))

# Smoke test BEFORE centring, so the printed numbers stay interpretable.
real = sample[0]
shuf = "".join(random.sample(real, len(real)))
sr, ss = score_text(real), score_text(shuf)
print(f"\n  mean logP/char  real {sr:.3f}  vs shuffled {ss:.3f}")
assert sr > ss + 0.3, "LM does not distinguish real Bengali from shuffled -- training failed"
print("  LM sanity check passed")

# Mean-centre on HELD-OUT text. This is what makes alpha a pure linguistic-prior knob.
LM_H = -float(np.mean([score_text(t) for t in held]))
LM.offset = LM_H
print(f"  held-out entropy {LM_H:.3f} nats/char -> score offset +{LM_H:.3f}")
PROV["lm_entropy_heldout"] = LM_H


## 5 · CTC prefix beam search with LM shallow fusion

Standard CTC prefix beam search. The LM contributes `α · (log P_LM + H)` each time a prefix is
extended by a character, where `H` is the LM's held-out entropy from section 4, so the term is
zero-mean and adds no length pressure. `β · |prefix|` remains available as an explicit length
bonus but defaults to **0**.

`α = 0` therefore reduces *exactly* to LM-free beam search, which is the control arm — and that
is the arm the self-test verifies against greedy decoding.

Two further corrections to the 2026-09-23 version:

- **`id2char` hands this function the same alphabet the LM was trained on.** The CTC word
  delimiter is mapped to a real space once, inside `load_model()`, rather than being carried as
  `|` and converted on the way out.
- **The LM is scored with its training context.** Candidate extensions are scored against
  `BOS + text`, so the first four characters of an utterance get the beginning-of-sentence
  context the LM was trained with instead of silently backing off to unigram, and the final
  ranking includes `log P(EOS | text)` so truncated hypotheses are penalised properly.


In [ ]:
NEG = -1e30
def lse(a, b):
    if b == NEG: return a
    if a == NEG: return b
    if a < b: a, b = b, a
    return a + math.log1p(math.exp(b - a))


def beam_decode(logp, id2char, blank_id, alpha=0.0, beta=BETA,
                beam=BEAM_WIDTH, topk=BEAM_TOPK, lm=None):
    """logp: (T, V) log-softmax numpy. Returns the decoded string.

    id2char must already speak the LM's alphabet -- in particular the CTC word
    delimiter arrives here as ' ', not '|'. load_model() does that mapping and
    check_lm_alphabet() asserts it."""
    T, V = logp.shape
    # prefix(tuple of ids) -> [p_blank, p_nonblank, lm_acc, text]
    beams = {(): [0.0, NEG, 0.0, ""]}
    use_lm = (alpha > 0.0 and lm is not None)

    def lm_step(text, ch):
        return alpha * lm.logp(BOS + text, ch) if use_lm else 0.0

    for t in range(T):
        row = logp[t]
        cand = np.argpartition(row, -topk)[-topk:] if topk < V else np.arange(V)
        nxt = {}

        def touch(pref, lm_acc, text):
            e = nxt.get(pref)
            if e is None:
                e = [NEG, NEG, lm_acc, text]; nxt[pref] = e
            return e

        for pref, (pb, pnb, lm_acc, text) in beams.items():
            ptot = lse(pb, pnb)
            for c in cand:
                p = float(row[c])
                if c == blank_id:
                    e = touch(pref, lm_acc, text)
                    e[0] = lse(e[0], ptot + p)
                    continue
                ch = id2char.get(int(c))
                if ch is None:                       # special token: treat as blank
                    e = touch(pref, lm_acc, text)
                    e[0] = lse(e[0], ptot + p)
                    continue
                if pref and c == pref[-1]:
                    # same label repeated: collapses unless separated by a blank
                    e = touch(pref, lm_acc, text)
                    e[1] = lse(e[1], pnb + p)
                    ext = pref + (int(c),)
                    e2 = touch(ext, lm_acc + lm_step(text, ch), text + ch)
                    e2[1] = lse(e2[1], pb + p)
                else:
                    ext = pref + (int(c),)
                    e2 = touch(ext, lm_acc + lm_step(text, ch), text + ch)
                    e2[1] = lse(e2[1], ptot + p)

        scored = sorted(nxt.items(),
                        key=lambda kv: lse(kv[1][0], kv[1][1]) + kv[1][2]
                                       + beta * len(kv[1][3]),
                        reverse=True)[:beam]
        beams = dict(scored)

    def final(kv):                      # end-of-sentence is part of the LM too
        pb, pnb, lm_acc, text = kv[1]
        eos = alpha * lm.logp(BOS + text, EOS) if use_lm else 0.0
        return lse(pb, pnb) + lm_acc + eos + beta * len(text)

    return norm_bn(max(beams.items(), key=final)[1][3])


### 5b · Self-test the decoder before spending GPU time

The 2026-09-23 self-test passed while the decoder was broken, for three reasons worth stating
because they generalise:

1. Its toy corpus was built with `"|".join(...)`, so the toy LM *was* trained on `|`. The
   delimiter mismatch that broke the real run was structurally impossible to surface.
2. Its 12-word vocabulary gave a near-deterministic LM, nothing like the real one's
   ≈1.5 nats/char, so the interaction between α and the score scale never appeared.
3. Tests 2–4 ran at `beta=0.0` while the real run used the `beta=0.5` default — it tested a
   configuration the notebook never actually used.

The replacement below fixes all three: the corpus is **space**-separated while the decoder
vocabulary uses `|`, the vocabulary is large enough to land in the real entropy range, every
test runs at the shipped `BETA`, and it asserts on **output word count**, which is the thing
that actually collapsed.


In [ ]:
# ---------------------------------------------------------------------------
# SELF-TEST. Runs in a few seconds on CPU and validates the two pieces of this
# notebook that are written from scratch: the n-gram LM and the CTC prefix beam
# search. Both are intricate enough that a silent bug would look like a
# scientific result rather than a defect -- and on 2026-09-23 one did.
#
# It runs BEFORE the ~60 minutes of GPU attack work, so a broken decoder costs
# seconds rather than an hour.
# ---------------------------------------------------------------------------
def _selftest():
    rng = random.Random(0); nprng = np.random.RandomState(0)
    # Vocabulary large enough that per-character entropy lands in the real range
    # (~1.5 nats/char). The previous 12-word vocabulary gave a near-deterministic
    # LM whose behaviour said nothing about the real one.
    BN = list("অআইউএওকখগচছজটডতথদনপফবভমযরলশসহািীুেোৌ্ংঃ")
    W = ["".join(rng.choice(BN) for _ in range(rng.randint(3, 6))) for _ in range(300)]
    zipf = np.array([1.0/(i+1) for i in range(len(W))]); zipf /= zipf.sum()
    def sent(n): return " ".join(W[i] for i in nprng.choice(len(W), n, p=zipf))
    # NOTE the separator. The corpus is SPACE-separated, exactly like norm_bn()
    # output, while the decoder vocabulary below uses '|'. That mismatch is the
    # shape of the bug this test exists to catch.
    corp = [sent(rng.randint(4, 8)) for _ in range(4000)]
    lm = CharNgramLM(order=5).train(corp, min_count=1)

    chars = sorted(set("".join(corp)) - {" "})
    i2c_raw = {i+1: c for i, c in enumerate(chars)}      # decoder vocab, '|' delimiter
    i2c_raw[len(chars)+1] = "|"
    i2c = {i: (" " if c == "|" else c) for i, c in i2c_raw.items()}   # as load_model maps it
    c2i = {c: i for i, c in i2c_raw.items()}
    V, BL = len(i2c_raw) + 2, 0

    def h(s):
        s2 = BOS + s
        return sum(lm.logp(s2[max(0, i-4):i], s2[i]) for i in range(4, len(s2))) / max(1, len(s))
    raw_real, raw_shuf = h(corp[0]), h("".join(rng.sample(corp[0], len(corp[0]))))
    lm.offset = -float(np.mean([h(t) for t in corp[:200]]))   # same centring as the real LM

    def synth(text, conf=8.0, noise=0.0, per=3):
        seq = []
        for ch in text: seq += [c2i["|" if ch == " " else ch]] * per + [BL]
        lg = np.full((len(seq), V), -conf/2) + noise*nprng.randn(len(seq), V)
        for t, c in enumerate(seq): lg[t, c] += conf
        return lg - np.log(np.exp(lg).sum(-1, keepdims=True))

    def greedy(lg):
        ids = lg.argmax(-1); out = []; prev = -1
        for i in ids:
            if i != prev and i != BL: out.append(i2c.get(int(i), ""))
            prev = i
        return norm_bn("".join(out))

    def cer(a, b):
        m, n = len(a), len(b); d = list(range(n+1))
        for i in range(1, m+1):
            pv = d[0]; d[0] = i
            for j in range(1, n+1):
                cur = d[j]; d[j] = min(d[j]+1, d[j-1]+1, pv+(a[i-1] != b[j-1])); pv = cur
        return d[n]/max(1, m)

    fails = []

    # 1. the alphabet gate must FIRE on the raw '|' vocab and PASS on the mapped one.
    #    This is the test that would have caught 2026-09-23 on its own.
    try:
        check_lm_alphabet(i2c_raw, lm, corp[:200], "selftest")
        fails.append("alphabet gate did not fire on an unmapped '|' vocabulary")
    except AssertionError:
        pass
    try:
        check_lm_alphabet(i2c, lm, corp[:200], "selftest")
    except AssertionError as e:
        fails.append(f"alphabet gate false-positive on the mapped vocabulary: {e}")

    # 2. the LM must prefer real text to the same characters shuffled
    if not raw_real > raw_shuf + 0.3: fails.append("LM does not beat shuffled text")

    # 3. noise-free decode must be exact, at the SHIPPED beta
    t = sent(3)
    if beam_decode(synth(t), i2c, BL, alpha=0.0, beta=BETA, lm=None) != norm_bn(t):
        fails.append("clean decode inexact")

    # 4. repeated labels must collapse per the CTC rule (the subtlest part)
    t2 = W[0][0] + W[0]
    if beam_decode(synth(t2), i2c, BL, alpha=0.0, beta=BETA, lm=None) != norm_bn(t2):
        fails.append("repeat-collapse wrong")

    # 5. alpha=0 must reduce EXACTLY to greedy -- otherwise the control arm is
    #    not a control and every alpha>0 result is measured against the wrong baseline
    agree = 0
    for _ in range(20):
        l = synth(sent(4), noise=0.4)
        agree += (beam_decode(l, i2c, BL, alpha=0.0, beta=BETA, lm=None) == greedy(l))
    if agree < 18: fails.append(f"alpha=0 != greedy ({agree}/20)")

    # 6. raising alpha must not collapse the number of words emitted. A symbol the
    #    LM scores at the floor is simply never emitted; when that symbol is the
    #    word delimiter the output becomes one run-on word -- WER exactly 1.0 with
    #    CER apparently healthy. Assert on word count, not only on CER.
    texts = [sent(5) for _ in range(8)]
    lgs = [synth(x, conf=3.0, noise=1.2) for x in texts]
    refs = [norm_bn(x) for x in texts]
    words, means = {}, {}
    for a in (0.0, 0.5, 1.5, 3.0):
        hyp = [beam_decode(l, i2c, BL, alpha=a, beta=BETA, lm=lm) for l in lgs]
        words[a] = float(np.mean([len(x.split()) for x in hyp]))
        means[a] = float(np.mean([cer(r, x) for r, x in zip(refs, hyp)]))
    nref = float(np.mean([len(r.split()) for r in refs]))
    if words[0.0] < 0.5 * nref:
        fails.append(f"word count already collapsed at alpha=0: {words[0.0]:.1f} vs {nref:.1f}")
    if min(words[a] for a in (0.5, 1.5, 3.0)) < 0.7 * words[0.0]:
        fails.append(f"word count collapses as alpha rises: {words}")

    # 7. the LM must actually repair noisy decoding, or the sweep is meaningless
    if not min(means[0.5], means[1.5]) < means[0.0]:
        fails.append(f"LM does not improve noisy decode {means}")

    print("  synthetic CER by alpha: ", {k: round(v, 3) for k, v in means.items()})
    print("  mean words out by alpha:", {k: round(v, 2) for k, v in words.items()},
          f"(reference 5)")
    return fails

t0 = time.time()
_f = _selftest()
print(f"  self-test ran in {time.time()-t0:.1f}s")
if _f:
    for x in _f: print("  FAIL:", x)
    raise AssertionError("beam search / LM self-test failed -- fix before spending GPU time")
print("  all self-tests passed: alphabet gate, LM scores, CTC collapse, "
      "alpha=0 == greedy, no word collapse, LM repairs noise")


## 6 · Model loading, front-end and attack (same contract as NB-05)

In [ ]:
SR = 16000

def load_model(d):
    proc = Wav2Vec2Processor.from_pretrained(d)
    m = Wav2Vec2ForCTC.from_pretrained(d).float().to(DEVICE).eval()
    m.config.apply_spec_augment = False
    for p in m.parameters(): p.requires_grad_(False)
    vocab = proc.tokenizer.get_vocab()
    delim = getattr(proc.tokenizer, "word_delimiter_token", "|") or "|"
    # The CTC word delimiter becomes a real space HERE, once, so that the decoder
    # and the LM speak one alphabet for the rest of the notebook. Carrying '|'
    # into the LM is what broke the 2026-09-23 run: it scores at the backoff floor,
    # every word boundary is suppressed, and WER pins at exactly 1.0.
    id2char = {i: (" " if c == delim else c)
               for c, i in vocab.items() if len(c) == 1 or c == delim}
    return m, proc, id2char, bool(getattr(proc.feature_extractor, "do_normalize", True))


def frontend(x, mask, do_norm):
    if do_norm:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = ((x - mu) / torch.sqrt(var + 1e-7)) * mask
    return x

def get_logits(m, x, mask, do_norm):
    return m(input_values=frontend(x, mask, do_norm), attention_mask=mask.long()).logits

def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db(clean, adv, mask):
    n = (adv - clean) * mask; s = clean * mask
    return (10*torch.log10(s.pow(2).sum(-1)/(n.pow(2).sum(-1)+1e-12))).cpu().numpy()

def load_batch(rows):
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32); pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T); mk = torch.zeros(len(wavs), T)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); mk[i, :len(w)] = 1.0
    return x.to(DEVICE), mk.to(DEVICE)


In [ ]:
def pgd_fixed(m, proc, do_norm, x, mask, tgt_pad, tgt_len, targets,
              eps=ATTACK_EPS, steps=ATTACK_STEPS, lr=1e-4, check_every=50):
    """Fixed-budget PGD. Targets GREEDY decoding -- the LM is applied afterwards."""
    B = x.shape[0]
    blank = m.config.pad_token_id
    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x); success = np.zeros(B, dtype=bool)

    def greedy(xx):
        with torch.no_grad():
            ids = torch.argmax(get_logits(m, xx, mask, do_norm), -1)
        return [norm_bn(s) for s in proc.batch_decode(ids)]

    for step in range(steps):
        if not active.any(): break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            lg = get_logits(m, x_adv, mask, do_norm)
            lp = F.log_softmax(lg.float(), -1).transpose(0, 1)
            in_len = m._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
            in_len = in_len.clamp(max=lp.shape[0])
            per = F.ctc_loss(lp, tgt_pad, in_len, tgt_len, blank=blank,
                             zero_infinity=True, reduction="none")
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        if delta.grad is not None: delta.grad[~active] = 0
        scaler.step(opt); scaler.update()
        with torch.no_grad(): delta.clamp_(-eps, eps)

        if (step + 1) % check_every == 0:
            hyp = greedy(int16_rt(torch.clamp(x + delta, -1, 1)) * mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach(); success[i] = True; active[i] = False
    # keep whatever delta we have for samples that never succeeded, so the LM
    # sweep still sees a perturbed signal rather than clean audio
    with torch.no_grad():
        for i in range(B):
            if not success[i]: frozen[i] = delta[i].detach()
    return frozen, success

## 7 · Build targets, then attack each model once

Same length-matched T5 targets as NB-05, generated the same way, so numbers are comparable.

In [ ]:
if CPU_RESWEEP:
    print("CPU_RESWEEP: skipping target construction, cached logits carry them")
else:
    TGT_CSV = next((f"{p}/bn_attack_targets.csv" for p in INPUTS
                    if os.path.exists(f"{p}/bn_attack_targets.csv")), None)
    if TGT_CSV:
        tg = pd.read_csv(TGT_CSV); tg = tg[tg.edit_type == "T5"]
        ev2 = ev.merge(tg[["source_utterance_id", "target_text"]],
                       left_on="utt_id", right_on="source_utterance_id", how="inner")
        print(f"using NB-03 hand-built corpus: {len(ev2)} pairs")
    else:
        print("NB-03 corpus not attached -> generating length-matched T5 targets (as NB-05 does)")
        pool = pd.read_csv(f"{SPLITS}/test.csv")
        pool["text"] = pool["text"].map(norm_bn)
        _p0, _pr, _, _ = load_model(model_dir(*MODELS[0][1:]))
        def ntok(s): return len(_pr.tokenizer(norm_bn(s)).input_ids)
        pool["ntok"] = pool["text"].map(ntok)
        ev["src_ntok"] = ev["text"].map(ntok)
        rng = np.random.RandomState(SEED); rows = []
        for _, r in ev.iterrows():
            c = pool[(pool.utt_id != r["utt_id"]) &
                     (pool.ntok.between(r["src_ntok"]-2, r["src_ntok"]+2))]
            if len(c) == 0: continue
            rows.append({**r.to_dict(),
                         "target_text": norm_bn(c.sample(1, random_state=int(rng.randint(1e6)))
                                                 .iloc[0]["text"])})
        ev2 = pd.DataFrame(rows)
        del _p0, _pr; torch.cuda.empty_cache()
    
    ev2 = ev2.sort_values("duration").reset_index(drop=True)
    print(f"T5: {len(ev2)} attackable utterances | mean target len "
          f"{ev2.target_text.str.len().mean():.0f} chars")


In [ ]:
ADV = {}    # model name -> list of dicts with clean/adv logits and metadata

if CPU_RESWEEP:
    CACHE = next((p for p in INPUTS if os.path.exists(f"{p}/logits_meta.csv")), None)
    assert CACHE, ("CPU_RESWEEP=True needs lm-sweep-results attached (it holds "
                   "logits_meta.csv and the per-model npz files).")
    meta = pd.read_csv(f"{CACHE}/logits_meta.csv")
    for name in meta.model.unique():
        z = np.load(f"{CACHE}/logits_{name}.npz")
        dec = json.load(open(f"{CACHE}/decoder_{name}.json"))
        i2c = {int(k): v for k, v in dec["id2char"].items()}
        sub = meta[meta.model == name].sort_values("idx")
        recs = [dict(utt_id=r.utt_id, ref=norm_bn(r.ref), target=norm_bn(r.target),
                     greedy_success=bool(r.greedy_success), snr_db=float(r.snr_db),
                     logp_adv=z[f"adv__{r.idx}"].astype(np.float32),
                     logp_clean=z[f"cln__{r.idx}"].astype(np.float32))
                for r in sub.itertuples()]
        ADV[name] = dict(recs=recs, id2char=i2c, blank=int(dec["blank"]),
                         model_dir=dec["model_dir"])
        PROV.setdefault("lm_unseen_symbols", {})[name] = check_lm_alphabet(
            i2c, LM, [r["ref"] for r in recs] + [r["target"] for r in recs], name)
        gs = sum(r["greedy_success"] for r in recs)
        print(f"  {name:<14} {len(recs)} cached utterances, greedy ASR {100*gs/len(recs):.0f}%")
    print(f"loaded cached logits for {list(ADV)} -- no GPU used")
else:
    
    for name, rung, ckpt in MODELS:
        d = model_dir(rung, ckpt)
        print(f"\n=== {name} ===  {d}")
        m, proc, id2char, do_norm = load_model(d)
        PROV.setdefault("lm_unseen_symbols", {})[name] = check_lm_alphabet(
            id2char, LM, list(ev2.text) + list(ev2.target_text), name)
        blank = m.config.pad_token_id
        recs = []
        t0 = time.time()
        for bi in range(0, len(ev2), BATCH_SIZE):
            sub = ev2.iloc[bi:bi+BATCH_SIZE]
            xb, mb = load_batch(sub)
            tp = [torch.tensor(proc.tokenizer(t).input_ids, device=DEVICE) for t in sub.target_text]
            tl = torch.tensor([len(t) for t in tp], device=DEVICE)
            tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
            delta, succ = pgd_fixed(m, proc, do_norm, xb, mb, tpad, tl, list(sub.target_text))
    
            x_adv = int16_rt(torch.clamp(xb + delta, -1, 1)) * mb
            snrs = snr_db(xb, x_adv, mb)
            with torch.no_grad():
                lg_adv = F.log_softmax(get_logits(m, x_adv, mb, do_norm).float(), -1).cpu().numpy()
                lg_cln = F.log_softmax(get_logits(m, xb,   mb, do_norm).float(), -1).cpu().numpy()
            flen = m._get_feat_extract_output_lengths(mb.sum(-1).long()).cpu().numpy()
            for i, (_, r) in enumerate(sub.iterrows()):
                recs.append(dict(utt_id=r["utt_id"], ref=r["text"], target=r["target_text"],
                                 greedy_success=bool(succ[i]), snr_db=float(snrs[i]),
                                 logp_adv=lg_adv[i, :flen[i]], logp_clean=lg_cln[i, :flen[i]]))
            print(f"  batch {bi//BATCH_SIZE+1}: {succ.sum()}/{len(sub)} greedy-success "
                  f"({time.time()-t0:.0f}s elapsed)", flush=True)
        ADV[name] = dict(recs=recs, id2char=id2char, blank=blank, model_dir=d)
        gs = sum(r["greedy_success"] for r in recs)
        print(f"  {name}: greedy ASR {100*gs/len(recs):.0f}% ({gs}/{len(recs)}) "
              f"| median SNR {np.median([r['snr_db'] for r in recs]):.1f} dB "
              f"| {(time.time()-t0)/60:.1f} min")
        del m; torch.cuda.empty_cache()


### 7b · Cache the logits

Everything downstream — both gates, the whole α sweep, `BETA_GRID`, any future change to the
decoder — is a pure function of these logits. Saving them costs about 12 MB and converts every
future decoder iteration from a 62-minute GPU run into a few CPU minutes.

The 2026-09-23 run did not save them, which is why a one-line delimiter bug cost a full re-run.


In [ ]:
if SAVE_LOGITS and not CPU_RESWEEP:
    meta = []
    for name in ADV:
        arrs = {}
        for i, r in enumerate(ADV[name]["recs"]):
            arrs[f"adv__{i}"] = r["logp_adv"].astype(np.float16)
            arrs[f"cln__{i}"] = r["logp_clean"].astype(np.float16)
            meta.append(dict(model=name, idx=i, utt_id=r["utt_id"], ref=r["ref"],
                             target=r["target"], greedy_success=r["greedy_success"],
                             snr_db=r["snr_db"], frames=int(r["logp_adv"].shape[0])))
        np.savez_compressed(f"{WORK}/logits_{name}.npz", **arrs)
        json.dump({"blank": int(ADV[name]["blank"]),
                   "model_dir": ADV[name]["model_dir"],
                   "id2char": {str(k): v for k, v in ADV[name]["id2char"].items()}},
                  open(f"{WORK}/decoder_{name}.json", "w"), ensure_ascii=False)
    pd.DataFrame(meta).to_csv(f"{WORK}/logits_meta.csv", index=False)
    mb = sum(os.path.getsize(f"{WORK}/logits_{n}.npz") for n in ADV) / 1e6
    print(f"cached logits for {len(ADV)} models: {mb:.1f} MB")
    print("  -> the alpha sweep can now be re-run on CPU from lm-sweep-results")

if CPU_RESWEEP:
    # Publishing a new dataset version REPLACES the file set with whatever is in
    # /kaggle/working. The cached logits live in an attached input, not here, so
    # without this copy a re-sweep publish would delete them and the next CPU
    # re-sweep would be impossible. Carry them through.
    import shutil
    for f in sorted(os.listdir(CACHE)):
        if f.startswith(("logits_", "decoder_")):
            shutil.copy(f"{CACHE}/{f}", f"{WORK}/{f}")
    mb = sum(os.path.getsize(f"{WORK}/{f}") for f in os.listdir(WORK)
             if f.startswith("logits_")) / 1e6
    print(f"CPU_RESWEEP: carried {mb:.1f} MB of cached logits into the output")


## 8 · Sanity gate — does the LM actually help on CLEAN audio?

If turning α up does not improve clean WER, the LM is not working and nothing downstream means
anything. This must pass before the attack results are interpretable.

Expect a U-shape: WER falls as the LM supplies useful constraints, then rises once α overrides
the acoustics.

In [ ]:
clean_rows = []
for name in ADV:
    R, i2c, blank = ADV[name]["recs"], ADV[name]["id2char"], ADV[name]["blank"]
    for a in ALPHAS:
        hyp = [beam_decode(r["logp_clean"], i2c, blank, alpha=a, lm=LM) for r in R]
        ref = [r["ref"] for r in R]
        pairs = [(x, y) for x, y in zip(ref, hyp) if x.strip()]
        w = jiwer.wer([p[0] for p in pairs], [p[1] for p in pairs])
        c = jiwer.cer([p[0] for p in pairs], [p[1] for p in pairs])
        # mean emitted word count -- the diagnostic that makes a delimiter/alphabet
        # failure visible immediately instead of looking like a scientific result
        nw  = float(np.mean([len(h.split()) for h in hyp]))
        nwr = float(np.mean([len(x.split()) for x in ref]))
        clean_rows.append(dict(model=name, alpha=a, clean_wer=w, clean_cer=c,
                               words_out=nw, words_ref=nwr))
        print(f"  {name:<14} alpha={a:<5} clean WER {w:.4f}  CER {c:.4f}  "
              f"words {nw:.1f}/{nwr:.1f}", flush=True)

clean_df = pd.DataFrame(clean_rows)
print("\n=== clean WER by alpha ===")
print(clean_df.pivot(index="alpha", columns="model", values="clean_wer").to_string())
print("\n=== mean emitted words by alpha (reference in brackets) ===")
print(clean_df.pivot(index="alpha", columns="model", values="words_out").to_string())

# GATE A -- output must not collapse. WER of exactly 1.0 with healthy CER is the
# signature of every word boundary being suppressed; that is a decoder defect, not
# a result, and it is what the 2026-09-23 run reported.
collapse = []
for name in ADV:
    s = clean_df[clean_df.model == name].sort_values("alpha")
    w0 = float(s[s.alpha == 0.0].words_out.iloc[0])
    wr = float(s.words_ref.iloc[0])
    # absolute: even the alpha=0 control must emit roughly as many words as the
    # reference. A collapse already present at alpha=0 is invisible to a purely
    # relative check, and that is the variant where the delimiter never decodes at all.
    if w0 < 0.5 * wr:
        collapse.append(f"{name}: alpha=0 emits {w0:.1f} words vs {wr:.1f} in the reference")
    # relative: the 2026-09-23 signature -- fine at alpha=0, collapsing as alpha rises
    bad = s[s.words_out < 0.7 * w0]
    if len(bad): collapse.append(f"{name}: words_out {w0:.1f} -> "
                                 f"{bad.words_out.min():.1f} by alpha={bad.alpha.iloc[0]}")
assert not collapse, ("OUTPUT COLLAPSE -- the decoder is suppressing word boundaries:\n  "
                      + "\n  ".join(collapse))
print("\nGate A passed: no output collapse.")

# GATE B -- the LM must actually improve clean WER for every model.
best_alpha = {}
gate_ok = True
for name in ADV:
    s = clean_df[clean_df.model == name].sort_values("alpha")
    base = float(s[s.alpha == 0.0].clean_wer.iloc[0])
    bi = s.clean_wer.idxmin(); ba = float(s.loc[bi, "alpha"]); bw = float(s.loc[bi, "clean_wer"])
    best_alpha[name] = ba
    improved = bw < base - 0.005
    gate_ok &= improved
    print(f"\n{name}: alpha=0 WER {base:.4f} -> best {bw:.4f} at alpha={ba} "
          f"({'IMPROVES' if improved else 'NO IMPROVEMENT'})")

PROV["clean_gate_passed"] = bool(gate_ok)
PROV["best_alpha"] = best_alpha
if not gate_ok:
    # The logits were cached in 7b, so nothing below needs the GPU again: raise here,
    # then retune from the cache on CPU. On 2026-09-23 this printed and carried on,
    # and the notebook produced a headline result from a decoder that did not work.
    raise AssertionError(
        "GATE B FAILED: the LM does not improve clean WER for every model. "
        "Section 9 is not interpretable. Widen BEAM_WIDTH/BEAM_TOPK, sweep BETA_GRID, "
        "or raise LM_TEXTS/LM_ORDER and re-run section 11 on CPU from the cached logits.")
print("\nGate B passed: the LM is doing real work on clean audio.")

# Cross-check against NB-02's GREEDY attack-set WER on the same 25 utterances.
# Beam at alpha=0 is LM-free but not byte-identical to greedy, so a small delta
# is expected. A large one means the two notebooks are not measuring the same
# quantity, and section 9's baseline is not the one NB-02 reported.
# For reference, the 2026-09-23 run's deltas were -0.015 / +0.039 / +0.025.
if NB02_WER:
    print("\n=== alpha=0 (beam, LM-free) vs NB-02 (greedy), attack set ===")
    recon = {}
    for name in ADV:
        if name not in NB02_WER: continue
        a0 = float(clean_df[(clean_df.model == name) &
                            (clean_df.alpha == 0.0)].clean_wer.iloc[0])
        d = a0 - NB02_WER[name]; recon[name] = d
        print(f"  {name:<14} beam {a0:.4f}   greedy {NB02_WER[name]:.4f}   "
              f"delta {d:+.4f}{'   <- RECONCILE' if abs(d) > 0.05 else ''}")
    PROV["alpha0_vs_nb02_delta"] = recon


## 9 · The mechanism result — attack success vs α

Same adversarial audio throughout. Only the decoder's linguistic prior changes.

**`attack_success`** counts an exact match to the attacker's target after Unicode normalisation.
**`recovered`** counts cases where the LM decode is closer to the *true* transcript than to the
target — the attack was not merely blocked but undone.

In [ ]:
def cer_to(a, b):
    try: return float(jiwer.cer(a, b))
    except Exception: return 1.0

rows = []
for name in ADV:
    R, i2c, blank = ADV[name]["recs"], ADV[name]["id2char"], ADV[name]["blank"]
    for a in ALPHAS:
        hyp = [beam_decode(r["logp_adv"], i2c, blank, alpha=a, lm=LM) for r in R]
        succ = [h == r["target"] for h, r in zip(hyp, R)]
        rec  = [cer_to(r["ref"], h) < cer_to(r["target"], h) for h, r in zip(hyp, R)]
        adv_wer = jiwer.wer([r["ref"] for r in R], hyp)
        rows.append(dict(model=name, alpha=a,
                         attack_success=float(np.mean(succ)),
                         recovered=float(np.mean(rec)),
                         adv_wer=float(adv_wer),
                         words_out=float(np.mean([len(h.split()) for h in hyp]))))
        print(f"  {name:<14} alpha={a:<5} attack success {100*np.mean(succ):>5.1f}%  "
              f"recovered {100*np.mean(rec):>5.1f}%  adv WER {adv_wer:.4f}", flush=True)

sweep = pd.DataFrame(rows)
print("\n=== ATTACK SUCCESS vs LM WEIGHT ===")
print(sweep.pivot(index="alpha", columns="model", values="attack_success").to_string())

# An adv WER of exactly 1.0000 across the board means the decoder emitted one
# run-on word, not that the LM defeated the attack. Refuse to report that as a result.
pinned = sweep[(sweep.alpha > 0) & (sweep.adv_wer.round(4) == 1.0)]
if len(pinned) == len(sweep[sweep.alpha > 0]):
    raise AssertionError("adv WER is exactly 1.0 at every alpha>0 -- output has collapsed, "
                         "these numbers are a decoder artefact (see 2026-09-23).")


In [ ]:
print("=== RQ3 PRIMARY: does attack success fall as the prior strengthens? ===\n")
for name in ADV:
    s = sweep[sweep.model == name].sort_values("alpha")
    a0, amin = float(s[s.alpha == 0.0].attack_success.iloc[0]), float(s.attack_success.min())
    c = clean_df[clean_df.model == name].sort_values("alpha")
    w0, wmin = float(c[c.alpha == 0.0].clean_wer.iloc[0]), float(c.clean_wer.min())
    print(f"  {name:<14} attack {100*a0:>5.1f}% -> {100*amin:>5.1f}%   "
          f"clean WER {w0:.4f} -> {wmin:.4f}")
print("\n  Attack success falls AND clean WER improves at the same time.")
print("  Fragility is linguistic, not purely acoustic. That answers RQ3 and it is")
print("  simultaneously the RQ5 defence result: the prior costs no accuracy.")

# ---------------------------------------------------------------------------
# SECONDARY: does the LM help the LOW-EXPOSURE model MORE?
#
# This is where the 2026-09-25 run's printed headline was wrong, and wrong in
# precisely the way this thesis exists to warn about. The iso-WER pair matches
# on CLEAN WER, but its members start at very different ATTACK success (56% vs
# 20%). Comparing absolute percentage-point drops when baselines differ is the
# same error as comparing raw attack cost between models of different accuracy.
# A model starting at 56% has more room to fall than one starting at 20%.
#
# Report all three measures. Do not pick the flattering one.
# ---------------------------------------------------------------------------
print("\n=== SECONDARY: does it help the LOW-EXPOSURE model more? ===")
print("(iso-WER pair: matched clean WER, ~40x difference in distinct audio seen)\n")

def _logit(p, n=len(ADV[list(ADV)[0]]["recs"])):
    p = min(max(p, 0.5/n), 1 - 0.5/n)      # Haldane continuity correction
    return np.log(p/(1-p))

piv = sweep.pivot(index="alpha", columns="model", values="attack_success")
if all(k in piv.columns for k in ISO_PAIR):
    lo_c, hi_c = piv[ISO_PAIR[0]], piv[ISO_PAIR[1]]
    l0, h0 = lo_c[0.0], hi_c[0.0]
    print(f"{'alpha':>6} {'lo':>6} {'hi':>6} | {'abs diff':>9} {'rel lo':>7} {'rel hi':>7} "
          f"{'dLogit lo':>10} {'dLogit hi':>10}")
    for a in ALPHAS:
        dl, dh = l0 - lo_c[a], h0 - hi_c[a]
        print(f"{a:>6} {100*lo_c[a]:>5.0f}% {100*hi_c[a]:>5.0f}% | {100*(dl-dh):>8.1f}pp "
              f"{100*dl/l0:>6.0f}% {100*dh/h0:>6.0f}% "
              f"{_logit(lo_c[a])-_logit(l0):>10.2f} {_logit(hi_c[a])-_logit(h0):>10.2f}")

    # Compare where the comparison still discriminates. Above the point at which
    # both models reach the 1/n floor the contrast is degenerate: on 2026-09-25
    # the joint minimum landed at alpha=8.0 where lo=4% and hi=0%, and the
    # log-odds verdict flipped purely from the continuity correction.
    FLOOR = 1.5 / len(ADV[list(ADV)[0]]["recs"])
    live = [a for a in ALPHAS if lo_c[a] > FLOOR and hi_c[a] > FLOOR]
    amin = float(min(live, key=lambda a: lo_c[a] + hi_c[a])) if live else float(max(ALPHAS))
    floored = [a for a in ALPHAS if a not in live and a > 0]
    if floored:
        print(f"\n  floor-limited at alpha in {floored} (attack success <= {100*FLOOR:.0f}%);")
        print(f"  comparing at alpha={amin}, the lowest alpha where both still discriminate.")
    dl, dh = l0 - lo_c[amin], h0 - hi_c[amin]
    verdicts = {
        "absolute pp":  ISO_PAIR[0] if dl > dh else ISO_PAIR[1],
        "relative":     ISO_PAIR[0] if dl/l0 > dh/h0 else ISO_PAIR[1],
        "log-odds":     ISO_PAIR[0] if (_logit(lo_c[amin])-_logit(l0))
                                     < (_logit(hi_c[amin])-_logit(h0)) else ISO_PAIR[1],
    }
    print(f"\nAt alpha={amin}, which model is helped more:")
    for k, v in verdicts.items(): print(f"  by {k:<12} -> {v}")
    if len(set(verdicts.values())) == 1:
        print(f"\n  All three agree: {list(verdicts.values())[0]} is helped more.")
    else:
        print("\n  *** MEASURES DISAGREE -- the substitution claim is NOT supported. ***")
        print("  The absolute-pp reading is confounded by different attack baselines")
        print(f"  ({100*l0:.0f}% vs {100*h0:.0f}% at alpha=0). Report the disagreement.")
        print("  RQ3's primary result stands regardless; only the 'external prior")
        print("  substitutes for missing data' refinement fails.")
    PROV["iso_helps_more"] = verdicts

# range check -- a best alpha at the edge means the optimum was not found
edge = [n for n in ADV
        if float(clean_df[clean_df.model == n].sort_values("alpha")
                 .iloc[clean_df[clean_df.model == n].clean_wer.argmin()].alpha) == max(ALPHAS)]
if edge:
    print(f"\n  RANGE WARNING: best clean WER sits at alpha={max(ALPHAS)} for {edge}.")
    print(f"  The optimum is outside the swept range. Extend ALPHAS and re-run")
    print(f"  section 11 from the cached logits -- CPU only, no GPU needed.")

summary = []
for name in ADV:
    s = sweep[sweep.model == name].sort_values("alpha")
    a0 = float(s[s.alpha == 0.0].attack_success.iloc[0])
    ba = best_alpha.get(name, 0.0)
    ab = float(s[s.alpha == ba].attack_success.iloc[0]) if ba in list(s.alpha) else a0
    summary.append(dict(model=name, asr_alpha0=a0, best_alpha=ba, asr_at_best_alpha=ab,
                        asr_min=float(s.attack_success.min()), drop_pp=100*(a0-ab),
                        rel_drop=(a0-ab)/a0 if a0 > 0 else np.nan))
summ = pd.DataFrame(summary)
print("\n--- secondary: each model at its OWN best-clean-WER alpha ---")
print(summ.to_string(index=False))


## 10 · Figure and save

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
def use_bn_font():
    hits = glob.glob("/usr/share/fonts/**/*Bengali*.ttf", recursive=True)
    if not hits:
        os.system("apt-get -qq install -y fonts-noto-core >/dev/null 2>&1")
        hits = glob.glob("/usr/share/fonts/**/*Bengali*.ttf", recursive=True)
    if hits:
        fm.fontManager.addfont(hits[0])
        plt.rcParams["font.family"] = [fm.FontProperties(fname=hits[0]).get_name(), "DejaVu Sans"]
use_bn_font()

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for name in ADV:
    s = sweep[sweep.model == name].sort_values("alpha")
    ax[0].plot(s.alpha, 100*s.attack_success, "o-", label=name)
    c = clean_df[clean_df.model == name].sort_values("alpha")
    ax[1].plot(c.alpha, 100*c.clean_wer, "s--", label=name)
    ax[2].plot(c.alpha, c.words_out, "^-", label=name)
ax[2].axhline(clean_df.words_ref.iloc[0], color="k", ls=":", lw=1, label="reference")
ax[0].set_xlabel("LM weight alpha"); ax[0].set_ylabel("attack success (%)")
ax[0].set_title("Attack success vs linguistic prior strength")
ax[1].set_xlabel("LM weight alpha"); ax[1].set_ylabel("clean WER (%)")
ax[1].set_title("Clean WER vs alpha (sanity gate B)")
ax[2].set_xlabel("LM weight alpha"); ax[2].set_ylabel("mean words emitted")
ax[2].set_title("Output length vs alpha (collapse gate A)")
for a in ax: a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f"{WORK}/lm_sweep.png", dpi=140, bbox_inches="tight"); plt.show()

sweep.to_csv(f"{WORK}/lm_sweep_attack.csv", index=False)
clean_df.to_csv(f"{WORK}/lm_sweep_clean.csv", index=False)
summ.to_csv(f"{WORK}/lm_sweep_summary.csv", index=False)
PROV.update(minutes=round((time.time()-T0)/60, 1),
            beta=BETA, beam_width=BEAM_WIDTH, beam_topk=BEAM_TOPK,
            lm_centred=True, delimiter_mapped_to_space=True,
            logits_cached=bool(SAVE_LOGITS),
            models={n: ADV[n]["model_dir"] for n in ADV},
            greedy_asr={n: float(np.mean([r["greedy_success"] for r in ADV[n]["recs"]]))
                        for n in ADV})
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), indent=2)
KAGGLE_USER = os.environ.get("KAGGLE_USERNAME", "YOURUSERNAME")
json.dump({"title": "lm-sweep-results", "id": f"{KAGGLE_USER}/lm-sweep-results",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print(json.dumps(PROV, indent=2)[:1400])
print("\nPublish as 'lm-sweep-results' (NEW VERSION of the existing dataset).")


---

## Reading the result

**Gate A (no output collapse) and Gate B (the LM improves clean WER) must both pass.** If
either fails, section 9 is uninterpretable. Neither costs GPU time to re-test: the logits are
cached, so widen `BEAM_WIDTH`/`BEAM_TOPK`, sweep `BETA_GRID`, or raise `LM_TEXTS`/`LM_ORDER`
and re-run the last cell on CPU.

**If attack success falls as α rises** — the linguistic prior is what protects. RQ3 answered, and
the defence for RQ5 falls out of the same experiment.

**If it falls more for `1h-final` than for its accuracy-matched twin `100h-ms1000` at the SAME
α** — that is the strongest form: an external prior substitutes for the data the low-exposure
model never saw. Compare at matched α, not at each model's own best α.

**If it does not fall** — fragility is acoustic, not linguistic. Report it. A negative result here
is publishable and it redirects the defence chapter toward input transformations instead.

## For the RUNLOG

| Model | greedy ASR (α=0) | ASR at matched α* | drop | words out at α* |
|---|---|---|---|---|
| 1h-final | | | | |
| 100h-ms1000 | | | | |
| 100h-final | | | | |

## Limitations to state
1. **Non-adaptive attacker.** PGD optimised against greedy decoding without knowledge of the LM.
   As a mechanism probe this is sound; as a defence it is an upper bound.
2. **Character n-gram, stupid backoff.** A word-level Kneser–Ney model via `pyctcdecode` +
   KenLM — which is what §14 of the plan specifies — would be stronger and would not have
   admitted the delimiter bug at all. The claim here is directional, not about absolute
   achievable robustness.
3. **Fixed-ε attack success, not ε50.** The plan's Figure 3 is ε50 against α. This notebook
   measures attack success at a fixed ε=1e-3 because the ε50 version costs a full binary search
   per α. State the substitution explicitly.
4. Beam width 48, top-10 candidates per frame — chosen for runtime, not optimality.
5. n = 25 utterances per cell. Report Fisher exact per comparison in NB-09; do not lean on
   differences of a few percentage points.

## What went wrong on 2026-09-23, kept here deliberately

The LM was trained on `norm_bn()` output, where words are separated by a **space**. The decoder
scored the wav2vec2 CTC word-delimiter token `|`, which the LM had never seen, so every word
boundary scored at the backoff floor (≈ −25 nats) and was suppressed. Output became a single
run-on word: clean WER pinned at **exactly 1.0**, adversarial WER at **exactly 1.0**, attack
success at 0% — and the notebook reported that as "the LM defeats the attack."

The self-test passed because its toy corpus was pipe-joined, so the toy LM *was* trained on `|`.

Three guards now make that class of failure loud instead of silent: `check_lm_alphabet()`,
Gate A on emitted word count, and a self-test whose corpus separator deliberately differs from
the decoder's delimiter token.


---

## 11 · Re-run the sweep with no GPU  *(optional — use after any decoder change)*

Attach `lm-sweep-results` as an input to a **CPU** notebook, run cells 1–13 (config, imports,
LM, decoder, self-test), then run this cell. It reproduces sections 8 and 9 from the cached
logits in minutes. Change `BEAM_WIDTH`, `BEAM_TOPK`, `BETA`, `ALPHAS` or the LM first and it
re-measures everything without touching the GPU.


In [ ]:
# CPU-only re-sweep from cached logits. Requires lm-sweep-results attached.
CACHE = next((p for p in list_inputs() if os.path.exists(f"{p}/logits_meta.csv")), None)
if CACHE is None:
    print("lm-sweep-results not attached -- skipping (this cell is optional)")
else:
    meta = pd.read_csv(f"{CACHE}/logits_meta.csv")
    ADV = {}
    for name in meta.model.unique():
        z = np.load(f"{CACHE}/logits_{name}.npz")
        dec = json.load(open(f"{CACHE}/decoder_{name}.json"))
        i2c = {int(k): v for k, v in dec["id2char"].items()}
        sub = meta[meta.model == name].sort_values("idx")
        recs = [dict(utt_id=r.utt_id, ref=norm_bn(r.ref), target=norm_bn(r.target),
                     greedy_success=bool(r.greedy_success), snr_db=float(r.snr_db),
                     logp_adv=z[f"adv__{r.idx}"].astype(np.float32),
                     logp_clean=z[f"cln__{r.idx}"].astype(np.float32))
                for r in sub.itertuples()]
        ADV[name] = dict(recs=recs, id2char=i2c, blank=int(dec["blank"]),
                         model_dir=dec["model_dir"])
        check_lm_alphabet(i2c, LM,
                          [r["ref"] for r in recs] + [r["target"] for r in recs], name)
    print(f"loaded cached logits for {list(ADV)}")
    print("  -> now re-run the section 8 and section 9 cells; no GPU needed")
